# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — The Anatomy of Growing Content

**Paper finding:**  
FlyRank compared content pages with rising versus falling impressions. The growing cohort contained 74,187 pages and the declining cohort contained 45,272 pages. Growing pages averaged about 3.2K words and 184 days of age, compared with 2.3K words and 230 days for declining pages. The paper reports that growing content was 37.6% longer and 20% younger than declining content. The paper also explicitly describes this as an observational comparison.

**Where does the label come from?**  
The growing/declining label is derived from the change in impressions between the current 30-day period and the previous 30-day period. Pages with more than 10% growth are classified as "up", while pages with more than 10% decline are classified as "down". Therefore, the label represents observed short-term impression movement rather than a subjective assessment of content quality.

**My methodology question:**  
Does this validation design support the stronger interpretation that greater content depth or younger content leads to future growth? The comparison demonstrates an association between page characteristics and observed trend direction, but it does not establish that changing word count or page age would cause future impression growth. A stronger design would evaluate the characteristics before the outcome period, ideally using a time-aware or matched comparison to test whether these characteristics are associated with subsequent growth.

**Constructive assessment:**  
The finding is useful as directional evidence about the structural characteristics of pages that were growing in this portfolio. I would keep the claim at the level of an observed association rather than treating the comparison as evidence that increasing word count will cause growth.

---

### Finding 2 — The Freshness Multiplier

**Paper finding:**  
FlyRank reports that 365+ day content refreshed within the previous 30 days showed a 3.2× higher Health Score, increasing from 10.7 to 34.5, and 57× more impressions, increasing from 71 to 4,039. The paper presents refresh timing as one of the strongest measured levers in the dataset.

**Where does the label/outcome come from?**  
The comparison separates older content according to whether it had been refreshed recently, and then compares Health Score and impressions between the groups. Freshness itself is defined as the number of days since the content was last updated. The paper's overall evidence is based on observational portfolio data rather than a randomized intervention.

**My methodology question:**  
Were the pages that were refreshed systematically different from the pages that were not refreshed before the refresh occurred? For example, were higher-quality, strategically important, or historically stronger pages more likely to be selected for refresh? If so, part of the observed difference could be due to selection effects rather than the refresh itself. A stronger validation design would compare refreshed pages with comparable unrefreshed pages using a pre-refresh baseline and a defined post-refresh observation window, ideally controlling for prior performance, age, topic, and other important differences.

**Constructive assessment:**  
The reported difference is a strong measured association in this portfolio, but the observational design does not by itself establish that refreshing a page causes a 3.2× Health Score increase or a 57× increase in impressions. I would therefore treat the result as directional decision-support evidence and validate it with a controlled or matched before/after comparison before making a causal claim.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Validation design

The Week-5 model was evaluated using a single client-group holdout. This prevented rows from the same client from appearing in both training and test sets, but the result could still depend on which clients happened to be selected for that one holdout.

For this audit, I use 5-fold GroupKFold validation with `client_id` as the grouping variable. Each client's rows remain together within a fold, so the model is evaluated on clients that were not present in that fold's training data.

I keep the Week-5 dataset, target, 20-feature set, Logistic Regression pipeline, and Precision@20 / Precision@50 metrics fixed. The validation design is the main change.

This validation answers a specific question: does the Week-5 model's ranking performance remain consistent across different unseen client groups, rather than depending on one particular client holdout?

I do not use a temporal split here because the 30,000-row starter snapshot does not expose a row-level observation date. A temporal experiment would require reconstructing historical feature windows and future outcome windows from the warehouse, which would constitute a separate modeling dataset rather than a controlled validation change.

In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parents[1]

data_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print(f"Loaded: {data_path}")

Loaded: c:\flyrank-ml-internship\data\raw\content_refresh_anonymized.csv


In [24]:
df['is_declining'] = (
  df["trend_direction"] == "down"
).astype(int)

In [26]:
group_col = "client_id"

print("Unique clients:", df[group_col].nunique())
print("\nRows per client:")
display(
    df.groupby(group_col)
      .size()
      .describe()
)

Unique clients: 32

Rows per client:


count      32.000000
mean      937.500000
std      1376.387113
min         3.000000
25%       110.250000
50%       567.000000
75%      1058.750000
max      7008.000000
dtype: float64

In [27]:
print("Missing client IDs:", df[group_col].isna().sum())

Missing client IDs: 0


In [5]:
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

target = "is_declining"

X = df[feature_cols]
y = df[target]
groups = df[group_col]

print("Features:", len(feature_cols))
print("Rows:", len(df))
print("Target rate:", y.mean())

Features: 20
Rows: 30000
Target rate: 0.5420666666666667


### 2.1 Week-5 model — original validation

The Week-5 experiment used the 30,000-row starter dataset with `is_declining` as the target and the following 20 features. The model was Logistic Regression with the preprocessing pipeline used in the Week-5 notebook.

The original validation used a client-group holdout. Complete clients were kept together so that pages from the same client did not appear in both the training and test sets.

The Week-5 result was:

- Precision@20: 0.85
- Precision@50: 0.84

The Week-4 baseline achieved Precision@20 = 0.40 and Precision@50 = 0.40 on the corresponding evaluation.

I reproduce this setup here as the "before" result. The purpose is not to retrain a different model, but to establish the result that the Week-6 validation audit will stress-test.

In [6]:
from sklearn.model_selection import GroupShuffleSplit

SEED = 42

gss = GroupShuffleSplit(
  n_splits = 1,
  test_size = 0.20,
  random_state = SEED
)

train_idx, test_idx = next(
  gss.split(
    df,
    y = df['is_declining'],
    groups = df["client_id"]
  )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train shape: ", train_df.shape)
print("Test shape: ", test_df.shape)

Train shape:  (23837, 45)
Test shape:  (6163, 45)


In [7]:
train_clients = set(train_df["client_id"])
test_clients = set(test_df["client_id"])

overlap = train_clients.intersection(test_clients)

print("Train Clients: ", len(train_clients))
print("Test Clients: ",len(test_clients))
print("Client Overlap: ", len(overlap))

assert len(overlap) == 0, "Client leakage detected!"

Train Clients:  25
Test Clients:  7
Client Overlap:  0


In [8]:
feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "avg_position",
    "ctr",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct"
]

X_train = train_df[feature_cols]
y_train = train_df["is_declining"]

X_test = test_df[feature_cols]
y_test = test_df["is_declining"]

In [9]:
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (23837, 20)
X_test: (6163, 20)
y_train: (23837,)
y_test: (6163,)


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = Pipeline([
  ("imputer", SimpleImputer(
    strategy = "median",
    add_indicator = True
  )),

  ("scaler", StandardScaler()),

  ("classifier", LogisticRegression(
    max_iter = 1000,
    random_state = 42
  ))
])

model.fit(X_train, y_train)

test_probs = model.predict_proba(X_test)[: ,1]

test_results = test_df[[
  "content_id",
  "client_id",
  "is_declining"
]].copy()

test_results["decline_probability"] = test_probs

test_results = test_results.sort_values(
  "decline_probability",
  ascending = False
)
test_results.head(10)

,content_id,client_id,is_declining,decline_probability
18063,content_b08562686d22,client_f369cb89fc,1,0.910564
3626,content_8ede62882d0b,client_f369cb89fc,1,0.908012
1537,content_a928cb66d230,client_f369cb89fc,1,0.901100
10175,content_374e795aab68,client_f369cb89fc,0,0.895327
27993,content_26d48a980581,client_f369cb89fc,0,0.886878
14741,content_2bc3b7c8b3d9,client_f369cb89fc,1,0.884278
8016,content_c94a53e3bfb8,client_f369cb89fc,0,0.881281
3195,content_b5e9e6453511,client_f369cb89fc,1,0.880531
23346,content_96dba8ca02c1,client_f369cb89fc,1,0.879457
2646,content_87c007fb5c26,client_f369cb89fc,1,0.876323


In [11]:
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

p20 = precision_at_k(
    test_probs,
    y_test,
    20
)

p50 = precision_at_k(
    test_probs,
    y_test,
    50
)

print("Precision@20:", p20)
print("Precision@50:", p50)

Precision@20: 0.85
Precision@50: 0.84


### 2.2 Week-6 validation design

The Week-5 result came from one client-group holdout, so its estimate could depend on the particular clients assigned to the test set.

For this audit, I use 5-fold GroupKFold validation with `client_id` as the grouping variable. This keeps every client's rows together while rotating which clients are held out for evaluation.

The model, target, 20 features, preprocessing, and evaluation metrics remain unchanged. The main change is the validation design.

GroupKFold does not test future-time generalization because the starter dataset does not provide a row-level observation date. Instead, it tests whether the Week-5 result is stable across multiple unseen client groups.

This provides a stronger check than relying on one client-held-out split while keeping the experiment comparable with Week 5.

In [12]:
from sklearn.model_selection import GroupKFold

gkf = GroupKFold(n_splits=5)

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):
    train_groups = set(groups.iloc[train_idx])
    test_groups = set(groups.iloc[test_idx])

    overlap = train_groups.intersection(test_groups)

    print(
        f"Fold {fold}: "
        f"train rows={len(train_idx):,}, "
        f"test rows={len(test_idx):,}, "
        f"train clients={len(train_groups)}, "
        f"test clients={len(test_groups)}, "
        f"client overlap={len(overlap)}"
    )

Fold 1: train rows=22,992, test rows=7,008, train clients=31, test clients=1, client overlap=0
Fold 2: train rows=24,269, test rows=5,731, train clients=25, test clients=7, client overlap=0
Fold 3: train rows=24,247, test rows=5,753, train clients=24, test clients=8, client overlap=0
Fold 4: train rows=24,245, test rows=5,755, train clients=24, test clients=8, client overlap=0
Fold 5: train rows=24,247, test rows=5,753, train clients=24, test clients=8, client overlap=0


The fold verification confirms that no client appears in both the training and test groups within any fold: all five folds have zero client overlap. The folds are not equal in row count because GroupKFold keeps complete clients together, and the clients have different numbers of rows. This is expected for grouped validation.

### 2.3 Training same LR with GroupKFold   

In [ ]:
gkf = GroupKFold(n_splits=5)

oof_prob = np.zeros(len(df))
oof_true = y.to_numpy()
fold_results = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X, y, groups=groups),
    start=1
):
    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y.iloc[train_idx]

    # Use the EXACT preprocessing/model configuration
    # from Week 5 here.
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator = True)),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    pipeline.fit(X_train, y_train)

    # Probability of the positive class: is_declining = 1
    oof_prob[test_idx] = pipeline.predict_proba(X_test)[:, 1]

    fold_p20 = precision_at_k(
        oof_prob[test_idx],
        y.iloc[test_idx].to_numpy(),
        20
    )

    fold_p50 = precision_at_k(
        oof_prob[test_idx],
        y.iloc[test_idx].to_numpy(),
        50
    )

    fold_results.append({
        "fold": fold,
        "test_rows": len(test_idx),
        "test_clients": len(set(groups.iloc[test_idx])),
        "precision_at_20": fold_p20,
        "precision_at_50": fold_p50,
        "base_rate": y.iloc[test_idx].mean()
    })

    print(
        f"Fold {fold}: "
        f"train={len(train_idx):,}, "
        f"test={len(test_idx):,}"
    )

fold_results_df = pd.DataFrame(fold_results)

display(fold_results_df)

print(
    f"Mean Precision@20: "
    f"{fold_results_df['precision_at_20'].mean():.3f}"
)

print(
    f"Std Precision@20: "
    f"{fold_results_df['precision_at_20'].std():.3f}"
)

print(
    f"Mean Precision@50: "
    f"{fold_results_df['precision_at_50'].mean():.3f}"
)

print(
    f"Std Precision@50: "
    f"{fold_results_df['precision_at_50'].std():.3f}"
)

Fold 1: train=22,992, test=7,008
Fold 2: train=24,269, test=5,731
Fold 3: train=24,247, test=5,753
Fold 4: train=24,245, test=5,755
Fold 5: train=24,247, test=5,753


,fold,test_rows,test_clients,precision_at_20,precision_at_50,base_rate
0,1,7008,1,0.75,0.70,0.490154
1,2,5731,7,0.40,0.48,0.645437
2,3,5753,8,0.75,0.84,0.379454
3,4,5755,8,0.90,0.72,0.622242
4,5,5753,8,0.95,0.94,0.584738


Mean Precision@20: 0.750
Std Precision@20: 0.215
Mean Precision@50: 0.736
Std Precision@50: 0.173


In [28]:
print("Rows:", len(df))
print("OOF predictions:", len(oof_prob))
print("Missing OOF predictions:", np.isnan(oof_prob).sum())

Rows: 30000
OOF predictions: 30000
Missing OOF predictions: 0


### 2.4 before vs after Model Comparison

In [29]:
comparison = pd.DataFrame({
    "validation": [
        "Week-5 single client holdout",
        "Week-6 5-fold GroupKFold"
    ],
    "precision_at_20": [
        0.85,
        fold_results_df["precision_at_20"].mean()
    ],
    "precision_at_20_std": [
        np.nan,
        fold_results_df["precision_at_20"].std()
    ],
    "precision_at_50": [
        0.84,
        fold_results_df["precision_at_50"].mean()
    ],
    "precision_at_50_std": [
        np.nan,
        fold_results_df["precision_at_50"].std()
    ]
})

display(comparison)

,validation,precision_at_20,precision_at_20_std,precision_at_50,precision_at_50_std
0,Week-5 single client holdout,0.85,NaN,0.840,NaN
1,Week-6 5-fold GroupKFold,0.75,0.215058,0.736,0.172858


- The Week-5 Logistic Regression achieved Precision@20 = 0.85 and Precision@50 = 0.84 under a single client-group holdout.

- Under five-fold GroupKFold validation, the mean fold-level Precision@20 was 0.75 (SD = 0.215), while mean Precision@50 was 0.736 (SD = 0.173).

- The grouped evaluation therefore produced lower average performance than the original single-holdout result, but the model retained substantial ranking performance on average. The fold-level results also varied considerably across client groups, with Precision@20 ranging from 0.40 to 0.95 and Precision@50 ranging from 0.48 to 0.94.

- This indicates that the Week-5 result was somewhat optimistic and that model performance is not uniform across clients. The GroupKFold evaluation provides a more robust view of cross-client performance, but it does not establish future-time performance because the starter snapshot does not provide the observation dates required for a temporal split.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3.1 Feature leakage audit

I audited the 20 features used by the Week-5 Logistic Regression model against three leakage categories:

1. **Label-derived:** the feature directly uses the target or information used to construct the target.
2. **Future/overlapping-window:** the feature contains information from the future outcome period or overlaps the period used to construct the label.
3. **Decision-derived:** the feature is created from a downstream action, intervention, product flag, or human decision that would not be independently available at prediction time.

The target is `is_declining`, which is derived from trend direction. Therefore, fields such as `trend_direction`, `trend_pct`, or other direct outcome-derived fields must not be included as model features.

The audit below checks the actual 20 Week-5 features rather than assuming that a feature is safe because it has a plausible name.

In [17]:
feature_audit = pd.DataFrame({
    "feature": feature_cols,
    "label_derived": [
        "No", "No", "No", "No", "No",
        "No", "No", "No", "No", "No",
        "No", "No", "No", "No", "No",
        "No", "No", "No", "No", "No"
    ],
    "future_or_overlapping": [
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "No apparent future dependency",
        "No apparent future dependency",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "No apparent future dependency",
        "No apparent future dependency",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot",
        "Cannot verify from snapshot"
    ],
    "decision_derived": [
        "No", "No", "No", "No", "No",
        "No", "No", "No", "No", "No",
        "No", "No", "No", "No", "No",
        "No", "No", "No", "No", "No"
    ]
})

display(feature_audit)

,feature,label_derived,future_or_overlapping,decision_derived
0,impressions_90d,No,Cannot verify from snapshot,No
1,clicks_90d,No,Cannot verify from snapshot,No
2,sessions_90d,No,Cannot verify from snapshot,No
3,pageviews_90d,No,Cannot verify from snapshot,No
4,users_90d,No,Cannot verify from snapshot,No
5,engaged_sessions_90d,No,Cannot verify from snapshot,No
6,days_with_impressions,No,Cannot verify from snapshot,No
7,days_with_sessions,No,Cannot verify from snapshot,No
8,content_age_days,No,No apparent future dependency,No
9,days_since_last_update,No,No apparent future dependency,No


In [18]:
target_related = [
    col for col in df.columns
    if col in ["is_declining", "trend_direction", "trend_pct"]
]

print("Target/outcome-related columns present:")
print(target_related)

print("\nWeek-5 features containing target-related fields:")
print([
    col for col in feature_cols
    if col in ["is_declining", "trend_direction", "trend_pct"]
])

Target/outcome-related columns present:
['trend_direction', 'trend_pct', 'is_declining']

Week-5 features containing target-related fields:
[]


The Week-5 feature list contains none of the direct target or target-construction fields identified in the dataset. In particular, `is_declining`, `trend_direction`, and `trend_pct` were excluded from the model features.

This rules out direct label leakage through those fields. It does not, by itself, prove that every aggregate feature is temporally safe, because the starter snapshot does not expose the underlying observation windows.

In [19]:
suspicious_keywords = [
    "future",
    "next",
    "trend",
    "label",
    "target",
    "declin",
    "change",
    "delta"
]

suspicious_columns = [
    col for col in df.columns
    if any(keyword in col.lower() for keyword in suspicious_keywords)
]

print("Potentially suspicious columns:")
print(suspicious_columns)

Potentially suspicious columns:
['trend_direction', 'trend_pct', 'is_declining']


### Deliberate leakage test

To demonstrate the effect of label leakage, I intentionally add a feature that is equal to the target itself. This is not a candidate production feature; it is a controlled test showing why outcome-derived information cannot be included in a predictive model.

If the evaluation becomes artificially close to perfect, that confirms that a feature containing the outcome would invalidate the evaluation.

In [20]:
df["deliberate_leak"] = df["is_declining"]

leaky_features = feature_cols + ["deliberate_leak"]

X_leaky = df[leaky_features]

oof_leak = np.zeros(len(df))

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(X_leaky, y, groups=groups),
    start=1
):
    X_train = X_leaky.iloc[train_idx]
    X_test = X_leaky.iloc[test_idx]
    y_train = y.iloc[train_idx]

    leak_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])

    leak_pipeline.fit(X_train, y_train)

    oof_leak[test_idx] = leak_pipeline.predict_proba(X_test)[:, 1]

print(
    "Leaky P@20:",
    precision_at_k(oof_leak, y.to_numpy(), 20)
)

print(
    "Leaky P@50:",
    precision_at_k(oof_leak, y.to_numpy(), 50)
)

Leaky P@20: 1.0
Leaky P@50: 1.0


In [21]:
del df["deliberate_leak"]

final_feature_cols = feature_cols.copy()

print("Final feature count:", len(final_feature_cols))
print("Leak feature included:", "deliberate_leak" in final_feature_cols)

Final feature count: 20
Leak feature included: False


The deliberate leakage feature is removed after the demonstration. The final model retains only the original 20 Week-5 features.

The artificially strong score from the leakage experiment is not treated as model performance. It demonstrates why a feature containing the outcome would invalidate the evaluation.

### Leakage audit conclusion

The audit found no direct use of the target or its explicit construction fields (`is_declining`, `trend_direction`, or `trend_pct`) among the original 20 model features.

The deliberate leakage experiment demonstrated that including a target-derived feature produces an artificially strong evaluation, confirming the importance of excluding outcome-derived information.

For future/overlapping-window leakage, the starter snapshot has an important limitation: it does not expose the underlying observation date and feature-window boundaries needed to independently reconstruct the historical 90-day windows relative to the decline label. I therefore do not claim that temporal leakage has been completely ruled out from the snapshot alone.

No Week-5 feature is explicitly a downstream product flag or action label based on the available feature definitions.

The final model therefore retains the original 20 features and excludes the deliberate leakage feature. Any stronger claim about exact feature-window timing would require reconstructing the feature and label windows from the dated warehouse data.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Logistic Regression model substantially outperforms the baseline and can effectively identify declining content.

### Evidence from the validation audit

The Week-5 client-held-out evaluation produced Precision@20 = 0.85 and Precision@50 = 0.84.

Under five-fold client-group validation, the mean fold-level Precision@20 was 0.75 (SD = 0.215) and mean Precision@50 was 0.736 (SD = 0.173). Performance varied substantially across folds, with Precision@20 ranging from 0.40 to 0.95 and Precision@50 ranging from 0.48 to 0.94.

### Safer claim

On the available 30,000-row starter dataset, the Logistic Regression model showed measured ranking performance under both a single client-held-out evaluation and five-fold client-group validation. Mean Precision@20 was 0.75 and mean Precision@50 was 0.736 under GroupKFold, compared with 0.85 and 0.84 in the original Week-5 holdout.

The lower GroupKFold averages and substantial fold-to-fold variation indicate that performance is sensitive to the client groups being evaluated. These results provide directional evidence that the model may be useful for prioritizing potentially declining content for human review, but they do not establish consistent performance across all clients or future-time performance. The model should therefore be treated as decision-support rather than as a definitive predictor of content decline.

The variation across client groups is an important part of the result: Precision@20 ranged from 0.40 to 0.95 across the five folds. This suggests that client-level differences may materially affect model performance and should be investigated before making broader generalization claims.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.